# Завдання 2 · Частина 2 — Оцінювання методом максимальної правдоподібності

Розглянемо процес реєстрації користувачів на сайті. Кожна спроба є випробуванням Бернуллі з невідомою ймовірністю успіху (конверсією) $p$. У наданому файлі `app_metrics.csv` статус реєстрації міститься в колонці `Registration_Success`: 1 — реєстрація, 0 — вихід.

Джерело: [assignment_2.pdf](../input/assignment_2.pdf) (в оригіналі — «Задача 4. Інференційна статистика»).

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

TASK_DIR = Path.cwd()
if (TASK_DIR / "labs" / "task2").is_dir():
    TASK_DIR = TASK_DIR / "labs" / "task2"
elif TASK_DIR.name == "notebooks":
    TASK_DIR = TASK_DIR.parent
INPUT_DIR = TASK_DIR / "input"
PDF_DIR = TASK_DIR / "pdf"
FIGURES_DIR = PDF_DIR / "figures"
SEED = 42
ALPHA = 0.05
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = PDF_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight"})


In [3]:
# Source: the app_metrics.csv supplied alongside assignment_2.pdf.
# Keep the source file unchanged; no external download is required.
DATA_PATH = INPUT_DIR / "app_metrics.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Place the supplied app_metrics.csv at {DATA_PATH}")
df = pd.read_csv(DATA_PATH)
required = {"Platform", "Session_Time", "Registration_Success", "Page_Views",
            "Purchase_Amount", "Revenue", "Subscription"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
for column in ("Registration_Success", "Subscription"):
    if not df[column].dropna().isin([0, 1]).all():
        raise ValueError(f"{column} must contain binary values 0 or 1")
# Review missing values before choosing a per-analysis handling policy.
display(df.head())
display(df.isna().sum().rename("missing"))

from scipy.optimize import minimize
from statsmodels.stats.diagnostic import lilliefors

P_BOUNDS = [(1e-5, 1 - 1e-5)]
FUTURE_VISITORS = 20


,User_ID,Platform,Session_Time,Registration_Success,Page_Views,Purchase_Amount,Revenue,Subscription
0,1001,Android,14.04,0,12,510.90,376.11,0
1,1002,Android,9.24,0,7,390.84,467.01,0
2,1003,iOS,17.76,0,13,640.27,228.87,0
3,1004,iOS,16.13,0,4,238.35,420.25,0
4,1005,Android,23.45,1,12,582.25,576.61,0


User_ID                 0
Platform                0
Session_Time            0
Registration_Success    0
Page_Views              0
Purchase_Amount         0
Revenue                 0
Subscription            0
Name: missing, dtype: int64

## 2(а). Перевірка нормальності реєстрацій

Завантажте колонку `Registration_Success` із `app_metrics.csv`. Виконайте одновибірковий критерій Колмогорова--Смирнова в модифікації Лілієфорса, коли параметри розподілу невідомі й оцінюються за вибіркою. Перевірте можливість нормального наближення цих бінарних даних. Запишіть $H_0$, $H_1$, статистику, p-значення та висновок.

In [4]:
registrations = df["Registration_Success"].dropna().to_numpy()
registration_d, registration_p = lilliefors(registrations, dist="norm", pvalmethod="table")
normality = pd.DataFrame({"n": [len(registrations)], "successes": [registrations.sum()],
                          "D": [registration_d], "p_returned": [registration_p],
                          "reject": [registration_p < ALPHA]})
display(normality)
normality.to_csv(RESULTS_DIR / "part2_normality.csv", index=False)

,n,successes,D,p_returned,reject
0,250,68,0.457078,0.001,True


***Answer***

У файлі 250 спостережень без пропусків; успішних реєстрацій 68. Припускаємо незалежність користувачів. За вимогою завдання перевіряємо $H_0$: розподіл спостережень нормальний з невідомими параметрами; $H_1$: він не є нормальним. Рівень значущості $\alpha=0.05$.

Виклик `lilliefors(registrations, dist="norm", pvalmethod="table")` повернув $D=0.457078$ та $p_{\mathrm{returned}}=0.001$. Це нижня межа табличної реалізації statsmodels: коректно записувати $p\le0.001$, а не вважати 0.001 точним p-значенням. Формально тест відхиляє $H_0$.

Водночас бінарна величина має лише значення 0 і 1, тому не може мати неперервний нормальний розподіл. Тест Лілієфорса розрахований на неперервні дані, тому для бінарних значень 0 і 1 його p-значення може бути неточним. Для окремої реєстрації використовуємо розподіл Бернуллі. Нормальне наближення може стосуватися вибіркової частки при великому обсязі, а не окремих нулів та одиниць.

## 2(б). Функція правдоподібності та чисельна оцінка

Запишіть функцію правдоподібності $L(p)$ і логарифмічну функцію $\ln L(p)$ для вибірки Бернуллі. Реалізуйте від'ємну логарифмічну правдоподібність $$-\ln L(p)=-\sum_{t=1}^{T}\bigl(r_t\ln p+(1-r_t)\ln(1-p)\bigr).$$ Знайдіть оптимальну оцінку за допомогою `scipy.optimize.minimize` з обмеженням $p\in[10^{-5},1-10^{-5}]$. Порівняйте чисельну оцінку $\widehat p_{MLE}$ з вибірковим середнім. У вихідній умові після формули NLL вжито слово «максимізуйте»; напрям оптимізації пояснено у відповіді.

In [5]:
def negative_log_likelihood(p_array):
    p = p_array[0]
    return -np.sum(registrations * np.log(p) + (1 - registrations) * np.log1p(-p))

fit = minimize(negative_log_likelihood, x0=[0.5], bounds=P_BOUNDS, method="L-BFGS-B")
if not fit.success:
    raise RuntimeError(fit.message)
p_hat = float(fit.x[0])
sample_proportion = float(registrations.mean())
mle_results = pd.DataFrame({"p_hat": [p_hat], "sample_mean": [sample_proportion],
                            "absolute_difference": [abs(p_hat - sample_proportion)],
                            "negative_log_likelihood": [fit.fun]})
display(mle_results)
mle_results.to_csv(RESULTS_DIR / "part2_mle.csv", index=False)

,p_hat,sample_mean,absolute_difference,negative_log_likelihood
0,0.272,0.272,3.254397e-08,146.309488


***Answer***

Нехай $r_t\in\{0,1\}$, $T=250$, $S=\sum_{t=1}^T r_t=68$. Для незалежних випробувань зі спільною ймовірністю $p$:
$$L(p)=\prod_{t=1}^{T}p^{r_t}(1-p)^{1-r_t}=p^{68}(1-p)^{182},$$
$$\ell(p)=\ln L(p)=68\ln p+182\ln(1-p),\qquad \operatorname{NLL}(p)=-\ell(p).$$
Максимізація $\ell(p)$ еквівалентна **мінімізації** NLL; слово «максимізуйте» після від’ємного логарифма в умові є неточністю. Аналітично
$$\ell'(p)=\frac{68}{p}-\frac{182}{1-p}=0\quad\Longrightarrow\quad\widehat p=\frac{68}{250}=0.272.$$
Друга похідна $\ell''(p)=-68/p^2-182/(1-p)^2<0$, тому це максимум. Чисельний мінімум `scipy.optimize.minimize` на $[10^{-5},1-10^{-5}]$ дає $\widehat p_{MLE}=0.27199997$; вибіркове середнє дорівнює 0.272. Абсолютна різниця $3.2544\times 10^{-8}$ зумовлена чисельною похибкою оптимізації. Оцінена конверсія становить 27.2%.

## 2(в). Біноміальні ймовірності майбутніх реєстрацій

За знайденою оцінкою $\widehat p$ обчисліть імовірності для наступних 20 відвідувачів:

- зареєструються рівно 5;

- зареєструються щонайменше 8;

- кількість реєстрацій буде строго більшою за $20\widehat p$.

Використайте PMF та CDF біноміального розподілу.

In [6]:
expected_registrations = FUTURE_VISITORS * p_hat
first_above_mean = int(np.floor(expected_registrations)) + 1
probabilities = pd.DataFrame({
    "event": ["K = 5", "K >= 8", f"K > 20*p_hat (K >= {first_above_mean})"],
    "probability": [stats.binom.pmf(5, FUTURE_VISITORS, p_hat),
                    stats.binom.sf(7, FUTURE_VISITORS, p_hat),
                    stats.binom.sf(first_above_mean - 1, FUTURE_VISITORS, p_hat)],
})
print(f"E(K) = {expected_registrations:.8f}; first integer above E(K): {first_above_mean}")
display(probabilities)
probabilities.to_csv(RESULTS_DIR / "part2_probabilities.csv", index=False)

E(K) = 5.43999935; first integer above E(K): 6


,event,probability
0,K = 5,0.197360
1,K >= 8,0.150328
2,K > 20*p_hat (K >= 6),0.472429


***Answer***

За припущення незалежності майбутніх відвідувачів і незмінної ймовірності реєстрації $K\sim\operatorname{Bin}(20,\widehat p)$. Використовуємо знайдену чисельну оцінку $\widehat p$:
$$P(K=5)=\binom{20}{5}\widehat p^5(1-\widehat p)^{15}=0.197360,$$
$$P(K\ge8)=1-F_K(7)=0.150328.$$
Оскільки $E(K)=20\widehat p\approx5.44$, строго більше за сподівання означає щонайменше 6 реєстрацій:
$$P(K>20\widehat p)=P(K\ge6)=1-F_K(5)=0.472429.$$
Ймовірності обчислено функціями `binom.pmf` та `binom.sf` (остання дорівнює $1-\mathrm{CDF}$). Це модельний прогноз із підставленою оцінкою $p$, який не враховує невизначеність самої оцінки та можливі зміни поведінки користувачів.